In [1]:
import os

In [2]:
%pwd

'd:\\PredictBot-Score-MLOps\\research'

In [3]:

# ".." tells Python to step out of the current folder into the parent folder
os.chdir("d:\\PredictBot-Score-MLOps")

# Check where you are now


In [4]:
%pwd

'd:\\PredictBot-Score-MLOps'

In [5]:
from src.predictor_bot_score.config.configuration import yaml_load , create_directories
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.constants import CONFIG_PATH
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import pandas as pd 
import io
import os
from src.predictor_bot_score.utils.src_util_s3_  import (save_manifest , 
                                                         save_local ,
                                                         self_local_save_mainfest , 
                                                         save_file_s3 ,
                                                         self_s3_mainfest,
                                                         s3_login)
from botocore.exceptions import  ClientError



In [6]:
@dataclass(frozen=True)
class DataTransformationConfig:
    validated_data_path: Path
    transformed_data_dir: Path
    bucket_name : str

In [ ]:
class Config_manager:

    def __init__(self , config = CONFIG_PATH):

        self.config_path = yaml_load(config)

        create_directories([self.config_path.artifacts_root])
    
    def get_data_transformation_config(self):

        config = self.config_path.data_transformation

        create_directories([config.transformed_data])

        data_transformation_config = DataTransformationConfig(
            validated_data_path=Path(config.validated_data_path),
            transformed_data =Path(config.transformed_data),
            bucket_name = self.config_path.s3_config.bucket_name
        )
        
        return data_transformation_config




In [12]:
class Data_transformation:

    def __init__(self, config : DataTransformationConfig):
        self.config = config
        self.BUCKET_NAME = self.config.bucket_name
        self.s3 = s3_login()
        self.data = self.read_data()
        self.pipeline_run_id = datetime.now().strftime("%Y_%m_%d_%H")
        

    def read_data(self):
        try:

            keys = []

            logger.info("=" * 50)
            logger.info("DATA TRANSFORMATION PIPELINE STARTED")
            logger.info("=" * 50)
            
            for page in self.s3.list_objects_v2(Bucket=self.BUCKET_NAME,Prefix='data_validation')['Contents']:
                if page.get('Key').endswith('.csv'):
                    keys.append(page.get('Key'))

            logger.info("S3 . Connection exists ")

            combined_file_key = sorted(keys)[-1]

            return combined_file_key
        
        except ClientError as e:

            # This is your custom message
            print("--- ALERT: The file is missing! Please check the path. ---")
            
            # This is the log file entry
            logger.error(f"File not found at: {self.BUCKET_NAME}")
            raise

    def transformed_data(self):
        
        try:
            file_key = self.data
            
            obj = self.s3.get_object(Bucket=self.BUCKET_NAME,Key=file_key)
            transformed_df = pd.read_csv(io.BytesIO(obj['Body'].read()))

            transformed_df["timestamp"] = pd.to_datetime(transformed_df["timestamp"], utc=True)

            transformed_df["bot_score"] = transformed_df["bot_score"].astype(float)

            return transformed_df
        
        except Exception as e:
            logger.info(e)
            raise

    def run(self):

        try:

            transformed_df = self.transformed_data()

            output_key = f"transformed_data/run__{self.pipeline_run_id}/transformed_data.csv"

            Manifest = save_manifest(pipline_id=self.pipeline_run_id,output_key=output_key)

            save_file_s3(df=transformed_df ,output_key=output_key ,BUCKET_NAME=self.BUCKET_NAME,s3_client=self.s3)

            self_s3_mainfest(manifest=Manifest ,output_key=output_key,BUCKET_NAME=self.BUCKET_NAME,s3_client=self.s3)

            save_local(df=transformed_df,local_dir=self.config.transformed_data,pipeline_run_id=self.pipeline_run_id)  ## df, local_dir

            self_local_save_mainfest(manifest=Manifest,local_dir=self.config.transformed_data,pipeline_run_id=self.pipeline_run_id)

            logger.info("=" * 50)
            logger.info("DATA TRANSFORMATION PIPELINE COMPLETED")
            logger.info("=" * 50)

        except Exception as e:
            logger.info(e)
            raise
    

In [13]:
transformation_config = Config_manager().get_data_transformation_config()
transformation_config = Data_transformation(transformation_config)
transformation_config.run()

[2026-07-11 10:43:45,775: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-11 10:43:45,780: INFO: common: Directory created (or already exists) at: artifacts]
[2026-07-11 10:43:45,782: INFO: common: Directory created (or already exists) at: artifacts/data_transformation/transformed_data]
[2026-07-11 10:43:45,791: INFO: 2646041408: ==================================================]
[2026-07-11 10:43:45,793: INFO: 2646041408: DATA TRANSFORMATION PIPELINE STARTED]
[2026-07-11 10:43:45,795: INFO: 2646041408: ==================================================]


[2026-07-11 10:43:47,078: INFO: 2646041408: S3 . Connection exists ]
[2026-07-11 10:43:57,704: INFO: src_util_s3_: Saved 40719 rows  s3://predict-bot-mlops/transformed_data/run__2026_07_11_10/transformed_data.csv]
[2026-07-11 10:43:58,176: INFO: src_util_s3_: Saved (manifest) rows  s3://predict-bot-mlops/transformed_data/run__2026_07_11_10/transformed_data.csv]
[2026-07-11 10:43:58,176: INFO: 2646041408: 'DataTransformationConfig' object has no attribute 'transformed_data']


AttributeError: 'DataTransformationConfig' object has no attribute 'transformed_data'

In [13]:
content = yaml_load(Path('config\config.yaml'))

[2026-07-10 18:54:13,417: INFO: common: yaml file: config\config.yaml loaded successfully]


In [20]:
content.s3_config.bucket_name

'predict-bot-mlops'